In [5]:
import pandas as pd
import ast
import os
import random
import M1 
import datetime

# 1. Configuration
OUT_DIR = "../data/output_data"
if not os.path.exists(OUT_DIR):
    os.makedirs(OUT_DIR)

string_matching_threshold = 0.5 

RESEARCHER_SKILLS_PATH = "../data/input_data/Set_4/large_researcher_skills.csv"
PROPOSAL_SKILLS_PATH   = "../data/input_data/Set_4/large_proposal_skills.csv"

# 2. Data Import 
# Load your uploaded data files
proposals_df = pd.read_csv(PROPOSAL_SKILLS_PATH) 
researchers_df = pd.read_csv(RESEARCHER_SKILLS_PATH) 

proposals_df['skills'] = proposals_df['skills'].apply(lambda x: ast.literal_eval(x))
researchers_df['skills'] = researchers_df['skills'].apply(lambda x: ast.literal_eval(x))

researcher_skills_dict = dict(zip(researchers_df['researcher_name'], researchers_df['skills']))
all_names = researchers_df['researcher_name'].tolist()

def generate_m1_teams_precise():
    results = []
    print("Start time:\t", datetime.datetime.now()) #
    
    for _, prop in proposals_df.iterrows():
        prop_link = prop['nsf_proposal_links_v0']
        prop_skills = list(prop['skills'])
        
        # Step 1: Generate Ranking AND Pseudo-Skills
        # pseudo_researcher_skills contains only the matched skills for THIS proposal
        ranking, pseudo_researcher_skills = M1.string_matching_ranking(
            all_researcher_skills=researcher_skills_dict,
            proposal_skills=prop_skills,
            pseudo_researcher_skills={}, 
            matching_threshold=string_matching_threshold
        )
        
        lead_researcher = random.choice(all_names)
        
        # Step 2: Create Team using Ranking
        teams_list = M1.create_teams_for_each_person(
            ranking=ranking,
            target_researcher=lead_researcher,
            num_of_teams=1
        )
        
        if not teams_list:
            continue
            
        team = teams_list[0]
        
        # Step 3: Apply Metric using Pseudo-Skills
        # IMPORTANT: We pass pseudo_researcher_skills instead of the full dict
        goodness = M1.apply_ultra_metric(
            set(prop_skills), 
            team, 
            pseudo_researcher_skills
        )
        
        results.append({
            'nsf_proposal_links_v0': prop_link,
            'lead_researcher': lead_researcher,
            'team': team,
            'goodness_score': goodness
        })
        
    print("End time:\t", datetime.datetime.now()) #
    return pd.DataFrame(results)

# 3. Execution
final_results = generate_m1_teams_precise()
output_path = os.path.join(OUT_DIR, 'teaming_results_m1_precise.csv')
final_results.to_csv(output_path, index=False)
print(f"Process complete. {len(final_results)} rows saved.")

Start time:	 2026-02-10 16:28:14.071525
End time:	 2026-02-10 16:32:19.958013
Process complete. 500 rows saved.
